# 00 - HYCOM Daily Resampling

Resamples raw HYCOM 3-hourly data to daily resolution for each year and saves separate files for temperature and salinity.

**Steps:**
1. Scan the HYCOM source folder for all 3-hourly netCDF files
2. Group files by year
3. Concatenate and resample to daily means (skipping NaNs)
4. Save per-year daily files: `hycom_water_temp_daily_{year}.nc` and `hycom_salinity_daily_{year}.nc`

In [ ]:
from collections import defaultdict
import xarray as xr
from pathlib import Path
from tqdm import tqdm
from pyproj import datadir

datadir.set_data_dir("/home/jupyter-daniela/.conda/envs/peru_environment/share/proj")

In [6]:
hycom_sources = Path("/home/jupyter-daniela/suyana/sources/hycom")
output_path = Path("/home/jupyter-daniela/suyana/peru_production/features")
output_path.mkdir(parents=True, exist_ok=True)

all_files = sorted(hycom_sources.rglob("hycom_*.nc"))

In [8]:
files_by_year = defaultdict(list)
for archivo in sorted(all_files):
    year = archivo.stem.split("_")[1][:4]
    files_by_year[year].append(archivo)

for year, files in tqdm(sorted(files_by_year.items()), desc="Processing years"):
    try:
        datasets = []
        for f in sorted(files):
            ds = xr.open_dataset(f)
            if "water_temp" not in ds or "salinity" not in ds:
                ds.close()
                continue
            datasets.append(ds)

        if not datasets:
            continue

        ds_year = xr.concat(datasets, dim="time").sortby("time")
        ds_daily = ds_year.resample(time="1D").mean(dim="time", skipna=True)

        ds_daily[["water_temp"]].to_netcdf(output_path / f"hycom_water_temp_daily_{year}.nc")
        ds_daily[["salinity"]].to_netcdf(output_path / f"hycom_salinity_daily_{year}.nc")

        for ds in datasets:
            ds.close()

        print(f"{year}: {ds_daily.sizes['time']} days saved")

    except Exception as e:
        print(f"{year}: ERROR - {e}")
        continue


Processing years:  10%|█         | 1/10 [00:15<02:15, 15.08s/it]

2015: 365 days saved


Processing years:  20%|██        | 2/10 [00:28<01:50, 13.85s/it]

2016: 366 days saved


/tmp/ipykernel_1071655/2770255258.py:19: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'lon' ('lon',) The recommendation is to set join explicitly for this case.
  ds_year = xr.concat(datasets, dim="time").sortby("time")
Processing years:  30%|███       | 3/10 [01:13<03:18, 28.35s/it]

...

2022: 365 days saved


Processing years:  90%|█████████ | 9/10 [03:53<00:26, 26.18s/it]

2023: 365 days saved


Processing years: 100%|██████████| 10/10 [04:07<00:00, 24.73s/it]

2024: 249 days saved
